# DevMind AI — Tokenizer Training

Train a custom Byte-Pair Encoding tokenizer on reviewed training data only. This is a research baseline; vocabulary size and tokenizer quality should be evaluated before model training.

In [ ]:
!pip -q install tokenizers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json
BASE = Path('/content/drive/MyDrive/DevMindAI/datasets')
TRAIN = BASE / 'processed' / 'train.jsonl'
AUDIT = BASE / 'audit' / 'formatted_pilot.jsonl'
TOKENIZER_DIR = Path('/content/drive/MyDrive/DevMindAI/models/tokenizer'); TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
print('Train file:', TRAIN, TRAIN.exists())
print('Formatted pilot:', AUDIT, AUDIT.exists())

## 1. Build a text iterator
Only reviewed training data should be used. Never include validation or test data in tokenizer training.

In [ ]:
def training_texts(path):
    if not path.exists(): raise FileNotFoundError(path)
    with path.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            obj = json.loads(line)
            if 'prompt' in obj and 'completion' in obj:
                yield obj['prompt'] + '\n' + obj['completion']
            else:
                yield '\n'.join(str(obj.get(k,'')) for k in ['instruction','input','response'])

SOURCE = TRAIN if TRAIN.exists() else AUDIT
print('Tokenizer source:', SOURCE)

## 2. Train BPE tokenizer
Start with a modest vocabulary. For small pilot corpora, a large vocabulary can overfit and waste model parameters.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
VOCAB_SIZE = 8000
SPECIAL_TOKENS = ['<pad>', '<unk>', '<bos>', '<eos>', '<system>', '<user>', '<assistant>']
tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, min_frequency=2, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
tokenizer.train_from_iterator(training_texts(SOURCE), trainer=trainer)
tokenizer.save(str(TOKENIZER_DIR / 'tokenizer.json'))
print('Tokenizer saved:', TOKENIZER_DIR / 'tokenizer.json')
print('Actual vocabulary:', tokenizer.get_vocab_size())

## 3. Smoke test
Check tokenization of code, stack traces, and natural language.

In [ ]:
samples = [
    'TypeError: unsupported operand type(s) for +',
    'def add(a, b):\\n    return a + b',
    'Explain the root cause and provide a corrected implementation.'
]
for sample in samples:
    encoded = tokenizer.encode(sample)
    print('TEXT:', sample)
    print('TOKENS:', encoded.tokens)
    print('IDS:', encoded.ids)
    print('DECODE:', tokenizer.decode(encoded.ids))
    print('-'*60)

## Next
After tokenizer inspection, Notebook 07 will implement a small decoder-only Transformer and verify tensor shapes before any training.